[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/03_class_weighting_vs_thresholds.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Class weighting versus moving the threshold

Allow about 30 minutes. Do we get a better review policy by using class weighting
during training or by simply shifting the decision threshold? We will train both
weighted and unweighted logistic regression models, debug an erroneous evaluation
that used `predict()`, tune the threshold using scikit-learn's cost-sensitive
threshold tuner, and finally, recover the probability scale from the weighted model.

The action is **send this recording for AF review**, and its cost is calculated in
reviews as in the core notebook, with the following values: $C_{TP}$ = 1, $C_{FP}$ =
1, $C_{FN}$ = 5, $C_{TN}$ = 0. Attempt the task on your own. Replace each `...` and
then run the checker below it. If you get stuck for about 10 minutes, you can use
the stuck switch below the problem (if present), or consult the worked copy in
`extension/solutions/`. If a later cell needs an unfinished problem, you will see
`Waiting for ...` in that cell. Once you have solved the problem or switched it,
click in the waiting cell, then choose **Runtime > Run before**, and then run the
waiting cell.

The dataset is not from CinC but instead consists of 2,000 synthetic recordings with
two artificially constructed features designed to mimic RR irregularity and absent P
waves. 200 of the recordings have AF. The features for each class are drawn from a
Gaussian distribution with the same covariance matrix. This ensures that the true
log-odds of AF are linear with respect to the features and that an unweighted
logistic regression is correctly specified. The dataset is split into train (1,000),
validation (500), and test (500) sets, each maintaining the same proportion of AF
recordings.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit, logit
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, make_scorer, roc_auc_score
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.utils.class_weight import compute_class_weight

from clinical_utility import features, reference
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
FEATURES = list(features.FEATURES)
rhythm = features.synthetic_rhythm_features()
train = rhythm[rhythm["split"] == "train"]
validation = rhythm[rhythm["split"] == "validation"]
test = rhythm[rhythm["split"] == "test"]

## What weighting asks the fit to learn

Because AF is uncommon, a common approach is to assign a higher weight `w` to each
AF training example compared to each example without AF. `class_weight="balanced"`
does this by rebalancing the classes to have the same total weight:

In [ ]:
compute_class_weight("balanced", classes=np.array([False, True]), y=train["af"])

For this dataset, the AF weight is 9 times that of each example without AF. Let
`p(x)` represent the unweighted probability of AF at a particular feature value `x`.
When training with weight `w` on AF, the optimal weighted output that maximises the
weighted training objective at `x` is `q = w*p / (w*p + 1 - p)`, that is

`logit(q) = logit(p) + log(w)`.

If the unweighted log-odds are linear in the features, the population optimum will
simply shift the intercept. In practice, with a finite sample, the slopes will be
slightly altered as well. For a weight ratio of 9, an unweighted probability of 0.1
would become a weighted output of 0.5. This means that despite the recording and its
features being identical, the weighted fit answers a different, reweighted question.
Consequently, the weighted model's outputs should not be interpreted as calibrated
probabilities for the original population.

Note that this explanation holds for the unregularised fit. scikit-learn's
`LogisticRegression` adds `L2` regularisation by default (`C=1`). Class weighting
scales the log-likelihood term relative to the regularisation penalty, meaning that
if you use the default settings, the intercept-only correction below will only be
approximately correct. We will use `C=np.inf`.

### Problem A: Make

Implement `fit_models(train)` to return a dictionary containing two trained models:
`"unweighted"`, a `LogisticRegression(C=np.inf, solver="newton-cholesky")`, and
`"weighted"`, the same model using `class_weight="balanced"`. Both models should be
trained on the training rows' features and labels `af`.

In [ ]:
def fit_models(train):
    X, y = train[FEATURES], train["af"]
    return {"unweighted": LogisticRegression(C=np.inf, solver="newton-cholesky").fit(X, y),
            "weighted": LogisticRegression(C=np.inf, solver="newton-cholesky",
                                           class_weight="balanced").fit(X, y)}
_ = check.run("weight_fit", fit_models)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# fit_models = check.use_reference("fit_models")

In [ ]:
if check.ready(fit_models=fit_models):
    models = fit_models(train)
    display(view.coefficients(models, FEATURES))
    a, b = models["unweighted"].coef_[0], models["weighted"].coef_[0]
    display(pd.DataFrame({
        "intercept shift": models["weighted"].intercept_[0] - models["unweighted"].intercept_[0],
        "log weight ratio": np.log((~train["af"]).sum() / train["af"].sum()),
        "cosine between slope vectors": a @ b / (np.linalg.norm(a) * np.linalg.norm(b)),
        "test AUC, unweighted": roc_auc_score(test["af"], models["unweighted"].predict_proba(test[FEATURES])[:, 1]),
        "test AUC, weighted": roc_auc_score(test["af"], models["weighted"].predict_proba(test[FEATURES])[:, 1]),
    }, index=["value"]).T.round(3))

The results show a shift in the model's intercept of 2.06, close to the log ratio of
weights (2.20). The slope vectors point in a near-identical direction (cosine
similarity: 0.995), with test AUCs of 0.898 and 0.896. The class weighting has
largely just changed the probability scale of the model's outputs and only modestly
changed the ranking of predictions here.

### Problem B: Fix the bug

A colleague compared the two models on the test set using `predict()` and concluded
that weighting the model improved sensitivity. Debug `test_report` in place to
correctly compare the two models, each at its own cutoff as determined by the priced
loss on the validation set (the row of `reference.operating_points` with the minimum
loss, tie-break by minimum number of reviews), and then apply each cutoff to the
test set's `predict_proba`.

In [ ]:
def test_report(models, validation, test, costs):
    """Each model's test reviews, AF detected and AF missed per 1,000, and its test loss."""
    rows = []
    for name, model in models.items():
        points = reference.operating_points(validation["af"],
                                            model.predict_proba(validation[FEATURES])[:, 1], costs)
        cutoff = points.sort_values(["loss", "reviews"], kind="stable").iloc[0]["threshold"]
        review = model.predict_proba(test[FEATURES])[:, 1] >= cutoff
        counts = reference.counts(test["af"], review)
        n = len(test)
        rows.append({"model": name, "cutoff": cutoff,
                     "reviews": 1000 * (counts["TP"] + counts["FP"]) / n,
                     "af_detected": 1000 * counts["TP"] / n, "af_missed": 1000 * counts["FN"] / n,
                     "loss": reference.policy_cost(test["af"], review, costs)})
    return pd.DataFrame(rows).set_index("model")


if check.ready(fit_models=fit_models):
    _ = check.show(test_report, fit_models(train), validation, test, costs)
_ = check.run("weight_report", test_report)

#### What predict() hid

`predict()` by default uses a decision boundary of 0.5 (exact ties are classified as
negative). We can see based on the test set performance that this means that 48
recordings per 1,000 will be reviewed with the unweighted model and 252 with the
weighted model. This seems to suggest that the weighted model is far more sensitive
(0.80 vs 0.30); however, this is not a fair comparison as the cutoff value of 0.5 is
the same numerical value but is on two separate probability scales and so reflects
different risk thresholds for the source population. If we instead compare the two
models when selecting the optimal cutoff for each based on the validation set, we
see that both models will review a similar number of recordings per 1,000 (174 vs
160) and that the unweighted model will actually miss slightly fewer AF per 1,000
(30 vs 32). Overall both models have similar losses on the test set (162 vs 160) and
so we can see that you can achieve almost the same priced policy with either a
class-weighted or unweighted model by just moving the threshold.

## Tune the threshold with scikit-learn

scikit-learn provides a class, `TunedThresholdClassifierCV`, that can be used to
tune the decision threshold of a classifier to optimise a particular metric. We can
leverage this with a scorer built from our loss with
`make_scorer(policy_cost, greater_is_better=False, costs=costs)`, so that it
minimises the priced loss. Note that the tuner's `cv=5` sets the number of
cross-validation folds: it splits the *training* rows five ways internally to choose
the threshold. This is separate from the train-validation split above, and the
validation set will not be used by the tuner. After fitting, the tuner automatically
applies the tuned threshold, not 0.5, when making predictions with the `.predict()`
method.

### Problem C: Make

Implement `tuned` by training a `TunedThresholdClassifierCV` model that wraps
`LogisticRegression(C=np.inf, solver="newton-cholesky")` and uses the scorer defined
above and `cv=5`. The model should be fit on the training set, using
`reference.policy_cost` as the loss.

In [ ]:
scorer = make_scorer(reference.policy_cost, greater_is_better=False, costs=costs)
tuned = TunedThresholdClassifierCV(LogisticRegression(C=np.inf, solver="newton-cholesky"),
                                   scoring=scorer, cv=5).fit(train[FEATURES], train["af"])
_ = check.run("weight_tuned", tuned)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# tuned = check.use_reference("tuned")

In [ ]:
if check.ready(tuned=tuned):
    review = tuned.predict(test[FEATURES]).astype(bool)
    summary = reference.operating_summary(test["af"], review)
    print("Tuned threshold:", round(tuned.best_threshold_, 3))
    print("Test reviews per 1,000:", 1000 * summary["reviews"] / len(test),
          " AF missed per 1,000:", 1000 * summary["af_missed"] / len(test),
          " test loss:", reference.policy_cost(test["af"], review, costs))

The tuned model threshold was 0.232, which is close to the value of 0.2 implied by
the prices. This is because the unweighted model's probabilities are close to being
calibrated. The resulting model reviews 130 recordings per 1,000 and misses 34 AF
per 1,000 on the test set, for a test loss of 150. Given the small number of AF
recordings in both the validation and test set (only 50 each), differences of a few
recordings between these policies are within sampling noise.

## Recover the probability scale

### Problem D: Make

As described above, the weighted model's log-odds predictions can be adjusted by
subtracting the log ratio of the class weights, `log(w)`. Implement
`undo_weighting(q, positive_to_negative_weight)` to perform this using `logit` and
`expit` from scipy. This is an idealised correction; it does not replace checking
calibration on held-out recordings.

In [ ]:
def undo_weighting(q, positive_to_negative_weight):
    return expit(logit(q) - np.log(positive_to_negative_weight))
_ = check.run("weight_correct", undo_weighting)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# undo_weighting = check.use_reference("undo_weighting")

In [ ]:
if check.ready(fit_models=fit_models, undo_weighting=undo_weighting):
    models = fit_models(train)
    weighted = models["weighted"].predict_proba(test[FEATURES])[:, 1]
    ratio = (~train["af"]).sum() / train["af"].sum()
    sigmoid = CalibratedClassifierCV(FrozenEstimator(models["weighted"]), method="sigmoid")
    sigmoid.fit(validation[FEATURES], validation["af"])
    display(pd.DataFrame({"test Brier": {
        "unweighted": brier_score_loss(test["af"], models["unweighted"].predict_proba(test[FEATURES])[:, 1]),
        "weighted, as output": brier_score_loss(test["af"], weighted),
        "weighted, corrected by log(w)": brier_score_loss(test["af"], undo_weighting(weighted, ratio)),
        "weighted, recalibrated on validation": brier_score_loss(
            test["af"], sigmoid.predict_proba(test[FEATURES])[:, 1])}}).round(4))

The weighted model's outputs have a Brier score of 0.1344 on the test set; however,
when the log-odds are corrected by subtracting `log(w)`, this improves to 0.0599.
This is in line with a model that had the weighted predictions recalibrated
empirically with a sigmoid function on the validation set (`CalibratedClassifierCV`
around a `FrozenEstimator` to prevent refitting of the model), with a Brier score of
0.0598. The unweighted model needed neither.

## Check your report

- It's important to distinguish between the training weights, the predicted
  probabilities and the threshold for review.
- Why does `predict()` make it seem like weighting produces a model with better
  detection?
- Where was each threshold set, and on which rows was it assessed?
- Under what circumstances is the log of the weight an exact vs an approximate
  correction?

**Question to consider:** if you have misspecified the log-odds or have a flexible
model with limited capacity, how may weighting impact the patterns that are learned?
What experiment could you run to test whether this is due to the model learning
different patterns versus simply an intercept shift?